In [27]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [28]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [29]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

13.03

In [30]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M1':mt5.TIMEFRAME_M1, 'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
    rates_frame['ema1'] =rates_frame['close'].ewm(span=9, adjust=False).mean()
    rates_frame['ema2'] =rates_frame['close'].ewm(span=15, adjust=False).mean()
    rates_frame['ema3'] =rates_frame['close'].ewm(span=50, adjust=False).mean()

#     rates_frame['ema'] =ema(rates_frame['close'], 9)
#     rates_frame['ema'] =ema(rates_frame['close'], 9)
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 7)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)
    
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
#         print(rates_frame.head())
    # Calculate Supertren
    return rates_frame

In [31]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [62]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 10000, 25, 'M15')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


for i in range(10, len(a)-1):
    if check==0:
#         print(str(a.iloc[i+1].name.time())[:6])
        if "00:00" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close < a.iloc[i].open:
            print("=="*20)
            print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2} !! {a.iloc[i].close}")
            c = 0
            buy_price = a.iloc[i].close
            pp_old = 0.0
            check=1
            checks = 0
            
#         if "00:05" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close > a.iloc[i].open:
#             print("=="*20)
#             print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2}")
#             c = 0
#             buy_price = a.iloc[i].close
#             check=2
#             continue
    elif check==1:
        sell_price = a.iloc[i].close
        lot = 1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
        ppopen = price_action(symbol, 3, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(3*10))
#         print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
#         if pp> 0.0:
#             checks=1
#         if pp < pp_old/1.5 and checks!=0:
#             print(f"PP_old {pp_old/2}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
#             profit.append(pp_old/2)
#             check =0
#             continue
        if c==0 and pp <-100:
#             if ppopen<0.0:
#             profit.append(ppopen)
            if ppopen< -500:
                profits.append(-500)
            else:
                profits.append(ppopen)
#             profit.append(pp)
            print(f"PPOPEN {ppopen}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            check =0
#             continue
        
#         if c==3 or pp < -5*(lot*10) or pp >=100*(lot*10):
#             print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
#             profit.append(pp)
#             check =0
#         pp_old = pp
        c+=1
        check =0

#     elif check==2:
#         sell_price = a.iloc[i].close
#         lot = 1
#         pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
# #         ppb = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
#         print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        
#         if c==6 or pp < -5*(lot*10) or pp >=100*(lot*10):
#             profit.append(pp)
#             check =0
#         c+=1

2024-06-25 23:45:00 !! 50.5612811876802 !! 54.16058605410528 !! 61889.22
2024-06-27 23:45:00 !! 38.45256087390551 !! 44.28398493136168 !! 61388.46
2024-06-30 23:45:00 !! 52.14569740312321 !! 57.39003904923725 !! 61884.63
2024-07-01 23:45:00 !! 43.54271040939666 !! 50.49432434699495 !! 63193.52
2024-07-03 23:45:00 !! 21.321665476426418 !! 30.17791810871202 !! 59530.54
PPOPEN -0.030000000000001137-- 45.78053659927738--- 41.81196104898038--59530.54 !! 59780.26--2024-07-04 00:00:00
2024-07-08 23:45:00 !! 47.24597422378019 !! 48.29608161653848 !! 56236.78
PPOPEN 894.27-- 52.824455476283475--- 50.50917941573365--56236.78 !! 56331.39--2024-07-09 00:00:00
2024-07-09 23:45:00 !! 57.510267104166594 !! 56.69867518484865 !! 57918.17
2024-07-13 23:45:00 !! 42.152059706994855 !! 47.9296342120689 !! 58616.26
2024-07-14 23:45:00 !! 61.540489118834365 !! 56.89199625529929 !! 60103.0
PPOPEN 1360.11-- 74.50710096043198--- 64.81755962779954--60103.0 !! 60297.3--2024-07-15 00:00:00
2024-07-16 23:45:00 !! 4

In [56]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 90000, 5, 'M15')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0

for i in range(10, len(a)-5):
#     print(str(a.iloc[i+1].name.time())[:6])
    if check==0:
        if "00:00" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close < a.iloc[i].open:
            print("=="*20)
            print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2} !! {a.iloc[i].close}")
            c = 0
            buy_price = a.iloc[i].close
            pp_old = 0.0
            check=1
            checks = 0
            
#         if "00:05" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close > a.iloc[i].open:
#             print("=="*20)
#             print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2}")
#             c = 0
#             buy_price = a.iloc[i].close
#             check=2
#             continue
    elif check==1:
        sell_price = a.iloc[i].close
        lot = 1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
        ppopen = price_action(symbol, 2, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
        print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        if pp> 0.0:
            checks=1
        if a.iloc[i].rsi1>a.iloc[i-1].rsi1:
            ppbet = price_action(symbol, 2, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
            print(f"PPBET {ppbet}-- { a.iloc[i+1].rsi1}--- {a.iloc[i+2].rsi1}--{ a.iloc[i+3].rsi1} !! {sell_price}--{a.iloc[i].name}")
            profit.append(ppbet)
            profits.append(ppbet)
            if ppbet <0.0:
                up =1
        if pp < pp_old/2 and checks!=0:
            print(f"PP_old {pp_old/2}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            profit.append(pp_old/2)
            check =0
            continue
        if c==0 and pp <-100:
#             if ppopen<0.0:
            profit.append(ppopen)
            profit.append(pp)
            print(f"PPOPEN {ppopen}-- { a.iloc[i+1].rsi1}--- {a.iloc[i+1].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i+1].name}")
            check =0
            continue
        
        if c==3 or pp < -5*(lot*10) or pp >=100*(lot*10):
            print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            profit.append(pp)
            check =0

        pp_old = pp
        c+=1

#     elif check==2:
#         sell_price = a.iloc[i].close
#         lot = 1
#         pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
# #         ppb = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
#         print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        
#         if c==6 or pp < -5*(lot*10) or pp >=100*(lot*10):
#             profit.append(pp)
#             check =0
#         c+=1

2022-02-26 23:45:00 !! 55.1542634127849 !! 56.335463650715035 !! 39409.01
PP 5.809999999999999-- 52.42566621461645--- 54.9375403687255--39409.01--2022-02-27 00:00:00
PP 176.02-- 38.740351963123246--- 47.215152239710065--39409.01--2022-02-27 00:15:00
PP 222.29-- 35.77830590849305--- 45.34899104569859--39409.01--2022-02-27 00:30:00
PP 245.87-- 34.22258343109874--- 44.38617251274099--39409.01--2022-02-27 00:45:00
PP 245.87-- 34.22258343109874--- 44.38617251274099--39409.01 !! 39141.14--2022-02-27 00:45:00
2022-02-27 23:45:00 !! 26.915882944752454 !! 28.78809981784576 !! 37396.09
PP -192.11-- 37.50588446935109--- 34.17974287730415--37396.09--2022-02-28 00:00:00
PPBET 96.78-- 40.98876056681758--- 51.677351420916--50.691206552339324 !! 37566.2--2022-02-28 00:00:00
PPOPEN 96.78-- 40.98876056681758--- 36.00156542926008--37396.09 !! 37566.2--2022-02-28 00:15:00
2022-03-05 23:45:00 !! 39.27921329757189 !! 48.51570664525636 !! 39333.38
PP 23.5-- 35.04576842266414--- 46.008992290938465--39333.38--

2022-04-27 23:45:00 !! 59.4179238934084 !! 54.878013640589465 !! 39079.42
PP -76.45-- 63.61886262532333--- 56.92497450399746--39079.42--2022-04-28 00:00:00
PPBET -58.72-- 61.12952011529717--- 73.4633503117078--61.40097618637844 !! 39133.87--2022-04-28 00:00:00
PP -76.45-- 63.61886262532333--- 56.92497450399746--39079.42 !! 39133.87--2022-04-28 00:00:00
2022-04-28 23:45:00 !! 41.588198400197534 !! 48.39979034661164 !! 39827.84
PP -170.78-- 50.73618005884128--- 52.49243718730922--39827.84--2022-04-29 00:00:00
PPBET -301.04-- 43.31458077033975--- 40.89124194423818--43.66590676988871 !! 39976.62--2022-04-29 00:00:00
PPOPEN -301.04-- 43.31458077033975--- 48.59963439545356--39827.84 !! 39976.62--2022-04-29 00:15:00
2022-04-30 23:45:00 !! 40.43637701403598 !! 42.947950938161554 !! 38313.31
PP 19.33-- 34.743606797201195--- 40.05721682639298--38313.31--2022-05-01 00:00:00
PP -18.13-- 43.19938550470973--- 43.75256587029319--38313.31--2022-05-01 00:15:00
PPBET 71.54-- 52.21802497734022--- 43.4671

2022-06-27 23:45:00 !! 60.42731108974498 !! 52.385098494135384 !! 20874.64
PP -16.310000000000002-- 58.66315101184399--- 51.790618111315595--20874.64--2022-06-28 00:00:00
PP 27.009999999999998-- 46.583380286831165--- 47.38198573484129--20874.64--2022-06-28 00:15:00
PP 75.12-- 36.77246106453858--- 43.00382712332427--20874.64--2022-06-28 00:30:00
PP 84.62-- 35.07085101111551--- 42.17510798510835--20874.64--2022-06-28 00:45:00
PP 84.62-- 35.07085101111551--- 42.17510798510835--20874.64 !! 20768.02--2022-06-28 00:45:00
2022-06-28 23:45:00 !! 23.895205099644357 !! 27.430854981682387 !! 20211.92
PP -110.04-- 46.493936162638754--- 37.361060480579106--20211.92--2022-06-29 00:00:00
PPBET -4.5-- 48.27487256917289--- 51.16392263313877--59.26296587816879 !! 20299.96--2022-06-29 00:00:00
PPOPEN -4.5-- 48.27487256917289--- 38.265227751545694--20211.92 !! 20299.96--2022-06-29 00:15:00
2022-06-30 23:45:00 !! 23.95130076461949 !! 31.22235211595762 !! 18690.69
PP -208.56-- 47.24814081158376--- 44.373159

2022-08-30 23:45:00 !! 58.15386090369705 !! 52.93855602013202 !! 19952.09
PP 24.82-- 51.276650261568435--- 49.731282087018734--19952.09--2022-08-31 00:00:00
PP 11.939999999999998-- 53.05830593515229--- 50.61763161038364--19952.09--2022-08-31 00:15:00
PPBET 20.259999999999998-- 56.12872277013333--- 43.818472593106904--44.238497157178585 !! 19918.15--2022-08-31 00:15:00
PP_old 12.41-- 56.12872277013333--- 50.61763161038364--19952.09 !! 19918.15--2022-08-31 00:15:00
2022-09-04 23:45:00 !! 54.551858040322664 !! 56.16716809511801 !! 19879.29
PP -12.48-- 50.49098845327643--- 54.51248689556449--19879.29--2022-09-05 00:00:00
PP -37.68-- 59.74516755812805--- 58.036606819636766--19879.29--2022-09-05 00:15:00
PPBET -110.9-- 43.14804552261773--- 41.53975864628331--36.47904744989155 !! 19894.97--2022-09-05 00:15:00
PP 6.77-- 43.14804552261773--- 50.59118593641023--19879.29--2022-09-05 00:30:00
PP 12.079999999999998-- 41.53975864628331--- 49.76977523105863--19879.29--2022-09-05 00:45:00
PP 12.079999

2022-11-03 23:45:00 !! 58.31218145898405 !! 53.35275403408438 !! 20252.42
PP 13.25-- 38.209863349367595--- 45.73020289114171--20252.42--2022-11-04 00:00:00
PP 15.25-- 37.35738955443453--- 45.334447003288595--20252.42--2022-11-04 00:15:00
PP 22.5-- 34.136476200887756--- 43.852900544176386--20252.42--2022-11-04 00:30:00
PP 39.25-- 27.699305866608995--- 40.55531395425606--20252.42--2022-11-04 00:45:00
PP 39.25-- 27.699305866608995--- 40.55531395425606--20252.42 !! 20191.17--2022-11-04 00:45:00
2022-11-05 23:45:00 !! 48.61948898627392 !! 51.07477953253293 !! 21325.26
PP -39.53-- 56.57931890991802--- 54.68371260653186--21325.26--2022-11-06 00:00:00
PPBET -65.0-- 46.3130606498347--- 36.801398250895275--36.38623627076435 !! 21342.79--2022-11-06 00:00:00
PP -18.03-- 46.3130606498347--- 49.828939795611085--21325.26--2022-11-06 00:15:00
PP 8.219999999999999-- 36.801398250895275--- 44.620357873972--21325.26--2022-11-06 00:30:00
PP 9.469999999999999-- 36.38623627076435--- 44.38244689551066--21325.

...

2023-10-14 23:45:00 !! 36.788328756637746 !! 40.106405686378096 !! 26838.22
PP -44.29-- 51.99151846364497--- 47.913404201419254--26838.22--2023-10-15 00:00:00
PPBET -138.94-- 29.948170566627084--- 37.68212325263505--35.03941785854536 !! 26860.51--2023-10-15 00:00:00
PP 14.18-- 29.948170566627084--- 35.01869859943861--26838.22--2023-10-15 00:15:00
PP -0.48999999999999844-- 37.68212325263505--- 39.423631910218155--26838.22--2023-10-15 00:30:00
PPBET -39.18-- 35.03941785854536--- 42.02000938685808--41.21871418064562 !! 26816.71--2023-10-15 00:30:00
PP_old 7.09-- 35.03941785854536--- 39.423631910218155--26838.22 !! 26816.71--2023-10-15 00:30:00
2023-10-16 23:45:00 !! 46.18968049218227 !! 53.274985310816966 !! 28404.82
PP 70.71-- 39.17594591514296--- 49.596574964415176--28404.82--2023-10-17 00:00:00
PP 21.549999999999997-- 44.398939874651276--- 51.50850665691636--28404.82--2023-10-17 00:15:00
PPBET 33.6-- 47.380025851685694--- 51.85806077018251--50.60501936094106 !! 28361.27--2023-10-17 00:

2023-12-17 23:45:00 !! 32.14491849670867 !! 43.64159552638981 !! 41882.31
PP 35.04-- 27.245571138809566--- 40.80532416577574--41882.31--2023-12-18 00:00:00
PP 151.38-- 19.994124850541994--- 35.70795215496213--41882.31--2023-12-18 00:15:00
PP 271.63-- 15.136220333586806--- 31.348896054085472--41882.31--2023-12-18 00:30:00
PP 196.34-- 27.92752923857033--- 36.56996424988519--41882.31--2023-12-18 00:45:00
PPBET -121.78-- 25.012921983809434--- 20.60026877784952--14.803452471508876 !! 41663.97--2023-12-18 00:45:00
PP 196.34-- 27.92752923857033--- 36.56996424988519--41882.31 !! 41663.97--2023-12-18 00:45:00
2023-12-20 23:45:00 !! 34.984500792876204 !! 42.268237510783294 !! 43444.04
PP -111.46-- 41.705419266743434--- 45.24168186438839--43444.04--2023-12-21 00:00:00
PPBET 250.3-- 50.74586711206593--- 47.05873554754005--47.62648967560049 !! 43533.5--2023-12-21 00:00:00
PPOPEN 250.3-- 50.74586711206593--- 49.50426744325708--43444.04 !! 43533.5--2023-12-21 00:15:00
2023-12-21 23:45:00 !! 77.235604

PP 40.85-- 56.7556127863393--- 56.156009555993684--49561.1--2024-02-14 00:00:00
PP 30.47-- 57.389868028578356--- 56.43793625162161--49561.1--2024-02-14 00:15:00
PPBET 70.72-- 60.41508739469887--- 58.8556194891721--52.738428311223025 !! 49508.63--2024-02-14 00:15:00
PP -15.89-- 60.41508739469887--- 57.744819657234466--49561.1--2024-02-14 00:30:00
PPBET -51.66-- 58.8556194891721--- 52.738428311223025--57.632942064936984 !! 49554.99--2024-02-14 00:30:00
PP_old 15.235-- 58.8556194891721--- 57.744819657234466--49561.1 !! 49554.99--2024-02-14 00:30:00
2024-02-15 23:45:00 !! 20.576854599393982 !! 30.371908448941625 !! 51377.77
PP -274.69-- 42.461462690306064--- 41.1161547614864--51377.77--2024-02-16 00:00:00
PPBET 227.72-- 50.34837829926878--- 46.04024502863656--46.674875322574074 !! 51630.46--2024-02-16 00:00:00
PPOPEN 227.72-- 50.34837829926878--- 45.58437949942283--51377.77 !! 51630.46--2024-02-16 00:15:00
2024-02-17 23:45:00 !! 77.38953044931964 !! 69.40271266667631 !! 51805.71
PP 73.49--

2024-04-29 23:45:00 !! 52.99415951671708 !! 53.86888089727552 !! 62927.36
PP 62.959999999999994-- 46.34717575192373--- 50.76376714908611--62927.36--2024-04-30 00:00:00
PP -4.140000000000001-- 51.90553386139077--- 53.06484098389835--62927.36--2024-04-30 00:15:00
PPBET 73.24-- 55.70499777642883--- 62.72652007042816--67.85684025370463 !! 62909.5--2024-04-30 00:15:00
PP_old 31.479999999999997-- 55.70499777642883--- 53.06484098389835--62927.36 !! 62909.5--2024-04-30 00:15:00
2024-05-01 23:45:00 !! 43.96552366380022 !! 46.533771450764235 !! 57268.08
PP -119.75-- 47.28071232905281--- 48.05793297222779--57268.08--2024-05-02 00:00:00
PPBET 1346.32-- 64.4534114621841--- 54.40517201978571--56.629334785709425 !! 57365.83--2024-05-02 00:00:00
PPOPEN 1346.32-- 64.4534114621841--- 57.244774774788745--57268.08 !! 57365.83--2024-05-02 00:15:00
2024-05-02 23:45:00 !! 29.504803358948877 !! 42.415160707739666 !! 58722.77
PP -89.29-- 34.451714034540345--- 44.46679529796276--58722.77--2024-05-03 00:00:00
PP

2024-07-20 23:45:00 !! 56.425476359933526 !! 60.78859496589102 !! 67381.46
PP 231.02-- 35.60142383652965--- 48.47651505379658--67381.46--2024-07-21 00:00:00
PP 265.8-- 33.61209267981006--- 47.065374576917186--67381.46--2024-07-21 00:15:00
PP 261.85-- 34.100002347877805--- 47.253171042867265--67381.46--2024-07-21 00:30:00
PPBET 65.86-- 39.83707421686453--- 51.12284966630196--42.27714798360435 !! 67097.61--2024-07-21 00:30:00
PP 217.92-- 39.83707421686453--- 49.40308177404889--67381.46--2024-07-21 00:45:00
PPBET 177.74-- 51.12284966630196--- 42.27714798360435--27.920040707955906 !! 67141.54--2024-07-21 00:45:00
PP 217.92-- 39.83707421686453--- 49.40308177404889--67381.46 !! 67141.54--2024-07-21 00:45:00
2024-07-21 23:45:00 !! 59.5707410713462 !! 58.441153902628 !! 67733.2
PP 55.56999999999999-- 56.35404162375707--- 56.72681010695631--67733.2--2024-07-22 00:00:00
PP -102.84-- 61.32909160041406--- 59.34933873782161--67733.2--2024-07-22 00:15:00
PPBET 551.44-- 68.8313718740174--- 71.2133258

In [63]:
n = 0
p = 0
tn = 0
tp = 0

for i in profits:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profits))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profits)}")

# print(time.time() - t1)
# -6, 6

1205.9800000000002
Total negative sm -->-1286.51
Total negative -->7
Total positive sm -->2492.4900000000002
Total positive -->3
Length 10


In [58]:
profits.sort()

In [59]:
profits

[-4834.1,
 -838.04,
 -837.2,
 -688.22,
 -629.66,
 -604.5,
 -575.68,
 -564.72,
 -545.42,
 -460.04,
 -453.44,
 -419.6,
 -414.72,
 -381.66,
 -363.44,
 -336.94,
 -304.72,
 -301.04,
 -300.4,
 -289.54,
 -288.04,
 -283.7,
 -281.72,
 -280.72,
 -280.1,
 -269.53999999999996,
 -264.96000000000004,
 -250.48,
 -248.24,
 -238.3,
 -238.24,
 -237.56,
 -233.82,
 -232.34,
 -203.48,
 -197.92,
 -197.2,
 -190.78,
 -189.5,
 -189.44,
 -180.72,
 -177.44,
 -170.18,
 -168.42,
 -164.5,
 -163.36,
 -162.9,
 -158.7,
 -157.48,
 -154.4,
 -151.76,
 -151.6,
 -144.45999999999998,
 -141.54000000000002,
 -140.12,
 -139.92000000000002,
 -138.94,
 -138.94,
 -136.66,
 -132.86,
 -132.78,
 -132.4,
 -131.28,
 -128.18,
 -125.82,
 -124.82,
 -123.62,
 -121.78,
 -117.68,
 -117.56,
 -112.92,
 -112.62,
 -111.14,
 -110.9,
 -110.28,
 -109.9,
 -108.64,
 -107.54,
 -106.44,
 -105.82,
 -105.04,
 -103.42,
 -101.32,
 -101.1,
 -100.52,
 -98.26,
 -97.68,
 -95.92,
 -94.68,
 -93.98,
 -92.34,
 -91.18,
 -89.92,
 -89.76,
 -89.36,
 -88.68,
 -88.0,
 

In [54]:
profit.sort()
profit

[-688.22,
 -628.3,
 -604.5,
 -564.72,
 -557.63,
 -545.42,
 -388.87,
 -359.59,
 -329.32,
 -322.13,
 -307.79,
 -283.7,
 -274.69,
 -271.72,
 -245.13,
 -242.75,
 -238.3,
 -223.34,
 -221.98,
 -216.3,
 -215.77,
 -205.91,
 -203.93,
 -202.12,
 -200.77,
 -197.2,
 -189.44,
 -183.71,
 -182.64,
 -177.64,
 -177.44,
 -174.67,
 -173.14,
 -168.32,
 -164.19,
 -164.15,
 -161.66,
 -160.9,
 -159.94,
 -156.4,
 -152.89,
 -148.81,
 -143.32,
 -140.26,
 -140.12,
 -139.92000000000002,
 -138.25,
 -136.66,
 -135.76999999999998,
 -135.03,
 -134.6,
 -126.41,
 -124.82,
 -124.41,
 -124.06,
 -119.75,
 -117.77,
 -117.56,
 -116.61,
 -116.55,
 -111.46,
 -111.14,
 -107.83,
 -107.38,
 -106.08,
 -105.56,
 -102.17,
 -97.34,
 -97.16,
 -96.57,
 -94.46,
 -92.87,
 -89.76,
 -89.29,
 -86.31,
 -85.62,
 -85.46000000000001,
 -85.25,
 -85.05,
 -82.5,
 -82.21000000000001,
 -82.02000000000001,
 -81.47999999999999,
 -79.36,
 -78.75999999999999,
 -77.18,
 -76.53,
 -76.03999999999999,
 -75.94,
 -74.0,
 -72.64,
 -71.83,
 -69.52000000000001,